# 02 — Feature Engineering

Build and visualise the feature set consumed by the ML ensemble.

**Topics covered:**
- Technical indicators (RSI, EMA, MACD, Bollinger, ATR, VWAP)
- Multi-timeframe alignment scores
- Key levels (support / resistance clustering)
- Session-tagged features
- Trend analyser output
- Feature correlation matrix & importance ranking

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import duckdb
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

DB_PATH = "../data/nexus.duckdb"

In [ ]:
conn = duckdb.connect(DB_PATH, read_only=True)
ohlcv = conn.execute(
    "SELECT * FROM ohlcv WHERE symbol='EURUSD' AND timeframe='M15' "
    "ORDER BY timestamp DESC LIMIT 500"
).fetchdf().sort_values('timestamp')
print(f"Candles: {len(ohlcv)}")
ohlcv.head()

In [ ]:
# Compute indicators using pandas_ta
import pandas_ta as ta

ohlcv.ta.rsi(length=14, append=True)
ohlcv.ta.ema(length=20, append=True)
ohlcv.ta.ema(length=50, append=True)
ohlcv.ta.macd(append=True)
ohlcv.ta.bbands(length=20, append=True)
ohlcv.ta.atr(length=14, append=True)

ohlcv.dropna().tail()

In [ ]:
# Feature correlation heatmap
feature_cols = [c for c in ohlcv.columns if c not in ('symbol', 'timeframe', 'timestamp')]
corr = ohlcv[feature_cols].corr()
fig = px.imshow(corr, text_auto='.2f', title='Feature Correlation Matrix', width=900, height=800)
fig.show()

In [ ]:
# S/R level clustering
from services.nexus_features.key_levels import compute_sr_levels  # adjust import as needed

# Placeholder: compute manually if import fails
highs = ohlcv['high'].values
lows = ohlcv['low'].values
print(f"Price range: {lows.min():.5f} — {highs.max():.5f}")

In [ ]:
conn.close()
print("Done.")